# 9. VLM checkpoint output analysis

Inspect **cached caption outputs** from notebook 8 before trusting aggregate CIDEr.
Checks alignment, degenerate generations, reference matching, side-by-side samples,
and per-image CIDEr outliers.

**Prerequisite:** run notebook 8 generation (section 8). Caption caches live in
`outputs/caption_eval/` locally and are mirrored to
`MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/` on Colab.

**Sections:**
1. Setup + locate caption caches
2. Config
3. Load predictions
4. Alignment (baseline vs AttnRes, same seed)
5. Automated red flags
6. Reference ↔ dataset verification
7. Side-by-side samples (eyeball)
8. Per-image CIDEr + outlier hunt
9. Verdict checklist

## 1. Setup

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


REPO = discover_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)

LOCAL_CACHE_DIR = REPO / "outputs" / "caption_eval"
DRIVE_CACHE_DIR = None
if _in_colab():
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_CACHE_DIR = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER / "outputs" / "caption_eval"


def caption_cache_roots() -> list[Path]:
    roots = [LOCAL_CACHE_DIR]
    if DRIVE_CACHE_DIR is not None:
        roots.append(DRIVE_CACHE_DIR)
    return roots


def resolve_caption_path(variant: str, seed: int) -> Path | None:
    name = f"{variant}_seed{seed}_captions.json"
    for root in caption_cache_roots():
        path = root / name
        if path.exists():
            return path
    return None


def resolve_summary_path() -> Path | None:
    for root in caption_cache_roots():
        path = root / "results_summary.json"
        if path.exists():
            return path
    return None


print("repo:", REPO)
print("local cache dir:", LOCAL_CACHE_DIR, "| exists =", LOCAL_CACHE_DIR.exists())
if DRIVE_CACHE_DIR is not None:
    print("drive cache dir:", DRIVE_CACHE_DIR, "| exists =", DRIVE_CACHE_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/AttnResGPT-next-scale
caption cache dir: /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval | exists = True


## 2. Config

In [3]:
SEEDS = [123, 456, 789]
VARIANTS = ["baseline", "attnres"]

# Must match notebook 8
DATASET_NAME = "Mozilla/flickr30k-transformed-captions"
DATASET_SPLIT = "train"
MAX_EXAMPLES = 20_000
MAX_GEN_TOKENS = 40

# Analysis knobs
EYEBALL_SEED = 456          # seed for side-by-side reading
N_EYEBALL = 15              # number of image pairs to print
PER_IMAGE_CIDER_SEED = 456  # seed for per-image outlier plot



## 3. Load predictions

In [4]:
import json


def load_records(path: Path) -> list[dict]:
    return json.loads(path.read_text())


def records_by_id(records: list[dict]) -> dict[int, dict]:
    return {int(r["image_id"]): r for r in records}


predictions: dict[tuple[str, int], dict[int, dict]] = {}
missing_files: list[str] = []

for variant in VARIANTS:
    for seed in SEEDS:
        path = resolve_caption_path(variant, seed)
        if path is None:
            missing_files.append(f"{variant}_seed{seed}_captions.json (not in local or drive cache dirs)")
            continue
        predictions[(variant, seed)] = records_by_id(load_records(path))
        print(f"loaded {variant} seed{seed}: {len(predictions[(variant, seed)])} images from {path}")

if missing_files:
    print("\nMISSING caption caches (run notebook 8 section 8 first):")
    for path in missing_files:
        print(" ", path)
elif len(predictions) == len(VARIANTS) * len(SEEDS):
    print("\nAll caption caches present.")


MISSING caption caches (run notebook 8 section 8 first):
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/baseline_seed123_captions.json
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/baseline_seed456_captions.json
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/baseline_seed789_captions.json
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/attnres_seed123_captions.json
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/attnres_seed456_captions.json
  /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs/caption_eval/attnres_seed789_captions.json


## 4. Alignment (baseline vs AttnRes)

In [5]:
for seed in SEEDS:
    b_key = ("baseline", seed)
    a_key = ("attnres", seed)
    if b_key not in predictions or a_key not in predictions:
        print(f"seed {seed}: SKIP (missing cache)")
        continue
    b_ids = set(predictions[b_key])
    a_ids = set(predictions[a_key])
    only_b = b_ids - a_ids
    only_a = a_ids - b_ids
    ref_mismatch = [
        i
        for i in b_ids & a_ids
        if predictions[b_key][i]["references"] != predictions[a_key][i]["references"]
    ]
    ok = not only_b and not only_a and not ref_mismatch
    print(
        f"seed {seed}: ids baseline={len(b_ids)} attnres={len(a_ids)} "
        f"only_b={len(only_b)} only_a={len(only_a)} ref_mismatch={len(ref_mismatch)} "
        f"=> {'OK' if ok else 'FAIL'}"
    )

seed 123: SKIP (missing cache)
seed 456: SKIP (missing cache)
seed 789: SKIP (missing cache)


## 5. Automated red flags

In [6]:
from collections import Counter

import pandas as pd


def caption_stats(records: dict[int, dict], *, variant: str, seed: int) -> dict:
    preds = [r["prediction"] for r in records.values()]
    word_counts = [len(p.split()) for p in preds]
    char_counts = [len(p) for p in preds]
    empty = sum(1 for p in preds if not p.strip() or p.strip() == ".")
    single_word = sum(1 for wc in word_counts if wc <= 1)
    dup_counts = Counter(preds)
    top_dup, top_dup_n = dup_counts.most_common(1)[0]
    longish = sum(1 for wc in word_counts if wc >= MAX_GEN_TOKENS - 2)
    return {
        "variant": variant,
        "seed": seed,
        "n": len(preds),
        "empty": empty,
        "single_word": single_word,
        "mean_words": round(sum(word_counts) / max(1, len(word_counts)), 2),
        "max_words": max(word_counts) if word_counts else 0,
        "near_cap": longish,
        "unique_preds": len(dup_counts),
        "top_dup_n": top_dup_n,
        "top_dup_preview": top_dup[:60],
    }


rows = [
    caption_stats(predictions[(v, s)], variant=v, seed=s)
    for v in VARIANTS
    for s in SEEDS
    if (v, s) in predictions
]
stats_df = pd.DataFrame(rows)
display(stats_df)

""


## 6. Reference ↔ dataset verification

Rebuild the notebook-8 val image set and confirm cached `references` match the dataset row.

In [7]:
from src.vlm.flickr30k import _row_captions, load_flickr30k_examples

VERIFY_SEED = EYEBALL_SEED


def expected_val_image_ids(seed: int) -> list[int]:
    dataset, train_records, val_records = load_flickr30k_examples(
        dataset_name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_examples=MAX_EXAMPLES,
        seed=seed,
    )
    train_ids = {r.row_index for r in train_records}
    unique_ids: list[int] = []
    seen: set[int] = set()
    for record in val_records:
        if record.row_index in train_ids or record.row_index in seen:
            continue
        seen.add(record.row_index)
        unique_ids.append(record.row_index)
    return unique_ids, dataset


if ("baseline", VERIFY_SEED) in predictions:
    expected_ids, dataset = expected_val_image_ids(VERIFY_SEED)
    cached_ids = sorted(predictions[("baseline", VERIFY_SEED)].keys())
    print(f"seed {VERIFY_SEED}: expected val images={len(expected_ids)} cached={len(cached_ids)}")
    print(f"  id sets match: {set(expected_ids) == set(cached_ids)}")

    import random

    random.seed(VERIFY_SEED)
    sample_ids = random.sample(cached_ids, min(5, len(cached_ids)))
    ref_errors = 0
    for image_id in sample_ids:
        cached_refs = sorted(predictions[("baseline", VERIFY_SEED)][image_id]["references"])
        row_refs = sorted(_row_captions(dataset[int(image_id)]))
        if cached_refs != row_refs:
            ref_errors += 1
            print(f"  MISMATCH image_id={image_id}")
    print(f"  reference spot-check errors: {ref_errors}/{len(sample_ids)}")
else:
    print(f"No baseline cache for seed {VERIFY_SEED}")

No baseline cache for seed 456


## 7. Side-by-side samples

Read these manually. Do both models describe the same scene reasonably?

In [8]:
seed = EYEBALL_SEED
b_key, a_key = ("baseline", seed), ("attnres", seed)

if b_key not in predictions or a_key not in predictions:
    raise RuntimeError(f"Missing caches for seed {seed}")

common_ids = sorted(set(predictions[b_key]) & set(predictions[a_key]))
step = max(1, len(common_ids) // N_EYEBALL)
sample_ids = common_ids[::step][:N_EYEBALL]

for image_id in sample_ids:
    b = predictions[b_key][image_id]
    a = predictions[a_key][image_id]
    refs = b["references"]
    print("=" * 72)
    print(f"image_id: {image_id}")
    print(f"refs[0]:  {refs[0] if refs else '(none)'}")
    if len(refs) > 1:
        print(f"refs[1]:  {refs[1]}")
    print(f"baseline:  {b['prediction']}")
    print(f"attnres:   {a['prediction']}")

RuntimeError: Missing caches for seed 456

In [ ]:
# Optional: show a few images in Colab (requires dataset download)
SHOW_IMAGES = False
N_IMAGES = 3

if SHOW_IMAGES and _in_colab():
    from IPython.display import display
    from src.vlm.flickr30k import _row_image

    _, dataset = expected_val_image_ids(EYEBALL_SEED)
    for image_id in sample_ids[:N_IMAGES]:
        img = _row_image(dataset[int(image_id)])
        print(f"image_id {image_id}")
        display(img)
        print("baseline:", predictions[("baseline", EYEBALL_SEED)][image_id]["prediction"])
        print("attnres: ", predictions[("attnres", EYEBALL_SEED)][image_id]["prediction"])
        print()

## 8. Per-image CIDEr + outliers

In [ ]:
import shutil

from pycocoevalcap.cider.cider import Cider

JAVA_OK = shutil.which("java") is not None


def per_image_cider(records: dict[int, dict]) -> dict[int, float]:
    cider = Cider()
    scores: dict[int, float] = {}
    for image_id, row in records.items():
        refs = [r for r in row["references"] if isinstance(r, str) and r.strip()]
        if not refs:
            continue
        pred = row["prediction"].strip() or "."
        gts = {image_id: [{"caption": r} for r in refs]}
        res = {image_id: [{"caption": pred}]}
        if JAVA_OK:
            from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

            ptb = PTBTokenizer()
            gts_t = ptb.tokenize(gts)
            res_t = ptb.tokenize(res)
        else:
            gts_t = {image_id: [c["caption"].lower().strip() for c in gts[image_id]]}
            res_t = {image_id: [c["caption"].lower().strip() for c in res[image_id]]}
        score, _ = cider.compute_score(gts_t, res_t)
        scores[image_id] = float(score[0])
    return scores


seed = PER_IMAGE_CIDER_SEED
if ("baseline", seed) in predictions and ("attnres", seed) in predictions:
    b_scores = per_image_cider(predictions[("baseline", seed)])
    a_scores = per_image_cider(predictions[("attnres", seed)])
    common = sorted(set(b_scores) & set(a_scores))
    diff_rows = [
        {
            "image_id": i,
            "cider_baseline": b_scores[i],
            "cider_attnres": a_scores[i],
            "delta_attnres_minus_baseline": a_scores[i] - b_scores[i],
        }
        for i in common
    ]
    diff_df = pd.DataFrame(diff_rows)
    print(f"seed {seed} mean per-image CIDEr: baseline={diff_df['cider_baseline'].mean():.4f} attnres={diff_df['cider_attnres'].mean():.4f}")
    print("\nWorst baseline (lowest CIDEr):")
    display(diff_df.nsmallest(5, "cider_baseline"))
    print("\nLargest attnres wins (positive delta):")
    display(diff_df.nlargest(5, "delta_attnres_minus_baseline"))
    print("\nLargest baseline wins (most negative delta):")
    display(diff_df.nsmallest(5, "delta_attnres_minus_baseline"))
else:
    print(f"Missing caches for per-image CIDEr seed {seed}")

## 9. Verdict checklist

In [ ]:
summary_path = resolve_summary_path()
if summary_path is not None:
    summary = json.loads(summary_path.read_text())
    print("Aggregate metrics from notebook 8 (results_summary.json):")
    for variant, agg in summary.get("aggregated", {}).items():
        cider = agg.get("CIDEr")
        if cider:
            print(f"  {variant}: CIDEr = {cider[0]:.4f} ± {cider[1]:.4f}")
else:
    print("No results_summary.json — run notebook 8 scoring section first.")

print("\n--- Manual verdict (fill in after reading section 7) ---")
print("[ ] image_id sets match between baseline and attnres per seed")
print("[ ] references match dataset rows (section 6)")
print("[ ] no mass empty / duplicate / single-word captions (section 5)")
print("[ ] side-by-side captions look plausible for the same image (section 7)")
print("[ ] if text looks similar but CIDEr diverges → investigate tokenization/scoring")
print("[ ] if attnres captions visibly worse → trust lower aggregate CIDEr")